In [ ]:

%load_ext ElasticNotebook

# Probability and Movies
### Looking at data from IMDB

#### Selena Flannery -- November 15, 2016

In [ ]:
%%RecordEvent
# %load_ext cudf.pandas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time



In [ ]:
%%RecordEvent
%%time
### cell 0 ###

filename = '/home/dias-benchmarks/new_notebooks/imdb/movie_metadata.csv'

m = pd.read_csv(filename)

In [ ]:
%%RecordEvent
%%time
### cell 1 ###

factor = 500
m = pd.concat([m] * factor)

In [ ]:
%%RecordEvent
%%time
### cell 2 ###

m.info()

In [ ]:
%%RecordEvent
%%time
### cell 3 ###

m.movie_title = m.movie_title.str.strip()
m.duration = pd.to_numeric(m.duration)
m.budget = pd.to_numeric(m.budget)
m.gross = pd.to_numeric(m.gross)
m.imdb_score = pd.to_numeric(m.imdb_score)
m.set_index("movie_title", inplace=True)
m.drop(["color", "director_facebook_likes", "actor_3_facebook_likes", "actor_2_name", "actor_1_facebook_likes", "actor_1_name"], axis=1, inplace=True)
m.drop(["cast_total_facebook_likes", "movie_imdb_link", "language", "actor_2_facebook_likes", "aspect_ratio"], axis=1, inplace=True)
m.drop(["actor_3_name", "facenumber_in_poster", "plot_keywords", "country"], axis=1, inplace=True)

In [ ]:
%%RecordEvent
%%time
### cell 4 ###

m.columns

## What is the probability that...

### A movie was longer than an hour and a half?

In [ ]:
%%RecordEvent
%%time
### cell 5 ###

ninety_min_num_movies = len(m.duration[m.duration > 90.0])
total_num_movies = len(m.index[m.duration != np.nan])
ninety_min_num_movies/total_num_movies

### A movie was longer than two hours?

In [ ]:
%%RecordEvent
%%time
### cell 6 ###

two_hour_movies = len(m.duration[m.duration > 120.0])
two_hour_movies/total_num_movies

### A movie was directed by Steven Spielberg?

In [ ]:
%%RecordEvent
%%time
### cell 7 ###

num_movies_directed = len(m.director_name[m.director_name != np.nan])
spiel = len(m.director_name[m.director_name == "Steven Spielberg"])
spiel/num_movies_directed

### A movie directed by Clint Eastwood will gross under budget?

In [ ]:
%%RecordEvent
%%time
### cell 8 ###

e_movies = m[m.director_name == "Clint Eastwood"]
e_gross_under_budget = len(e_movies[(e_movies.gross != np.nan) & (e_movies.budget != np.nan) & (e_movies.gross < e_movies.budget)])
e_gross_under_budget/len(e_movies.index)

### A movie generally grossed more than its budget?

In [ ]:
%%RecordEvent
%%time
### cell 9 ###

movies_with_budget_and_gross = m[(~pd.isnull(m.gross)) & (~pd.isnull(m.budget))]
gross_over_budget = m[(m.gross > m.budget) & (~pd.isnull(m.gross)) & (~pd.isnull(m.budget))]

len(gross_over_budget)/len(movies_with_budget_and_gross)


### A movie grossed over the average gross of this data set?

In [ ]:
%%RecordEvent
%%time
### cell 10 ###

average_gross = m.gross.mean()
movie_grossed_over_average = len(m.index[(~pd.isnull(m.gross)) & (m.gross > average_gross)])
total_movie_with_gross = len(m.index[~pd.isnull(m.gross)])
movie_grossed_over_average/total_movie_with_gross

## False Positives

###  A movie that was highly-rated but did poorly in the box office (gross < budget)

In [ ]:
%%RecordEvent
%%time
### cell 11 ###

movies_with_scores = m[(~pd.isnull(m.imdb_score) & (~pd.isnull(m.gross)) & (~pd.isnull(m.budget)))][["imdb_score", "gross", "budget"]]
positive_scores = movies_with_scores[movies_with_scores.imdb_score > 6]
false_positives = positive_scores[positive_scores.gross < positive_scores.budget]
len(false_positives)/len(positive_scores)

## False Negatives

### A movie that was poorly rated, but did well in the box office (gross > budget)

In [ ]:
%%RecordEvent
%%time
### cell 12 ###

negative_scores = movies_with_scores[movies_with_scores.imdb_score <= 6]
false_negatives = negative_scores[negative_scores.gross > negative_scores.budget]

len(false_negatives)/len(negative_scores)

### Tom Hanks vs Harrison Ford: Gross > budget

### Tom Hanks vs Harrison Ford: Ratings

# Visualizations

In [ ]:
%%RecordEvent
%%time
### cell 13 ###

scores = m.imdb_score

# plt.figure(figsize=(10, 3))
# plt.hist(scores, bins=np.arange(1, 11))
# plt.title("Distribution of Ratings")
# plt.xlabel("IMDB Rating")
# plt.ylabel("# of Movies")
# plt.show()

According to this graph, it appears that the distribution is skewed to the right, most films are rate at a 6 or a seven

In [ ]:
%%RecordEvent
%%time
### cell 14 ###

scores.describe()

In [ ]:
%%RecordEvent
%%time
### cell 15 ###

mean = scores.mean()
median = scores.quantile(.5)
std = scores.std()
mean, median, std

## What’s the probability that a movie’s length will be between 1:10 and 1:30?